# Notebook 03_1 — Predictive Performance (Text + Image Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Women's Shoes (Size 8)

---

Same as 03_2 but uses **multimodal embeddings** (RoBERTa + BEiT + SAINT).  
Adds `x_emb` as a fourth feature specification on top of x, x_pca, x_sim.

Model specifications compared:
- OLS / Boosting — Tabular only
- OLS / Boosting — Tabular + PCA
- OLS / Boosting — Tabular + Similarities
- OLS / Boosting — Tabular + Embeddings (multimodal)
- Deep Time Independent
- Deep Time Dependent

## ① Mount Drive

In [1]:
# Local mode - no Google Drive needed
print('Local mode')

Local mode


## ② Set Working Directory

In [2]:
import os, sys
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root")

CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print(f'Working directory: {os.getcwd()}')

Working directory: /home/iankuzuma/claude_code/demand-modeling-data-men-8/women-8-subcat-split/flats/code


## ③ Imports

In [3]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)
print('✅ Imports done')

✅ Imports done


## ④ Load Dataset

Loads the **multimodal** embeddings dataset from notebook 01_1 (txt_only=False).
Key difference from 03_2: this dataset includes both text AND image embeddings.

In [4]:
txt_only = False
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"

df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val   = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val   = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val   = df_full_val.dropna()

dummy_subcat_names = [category for category in df_full_val["subcat_aggregated"].unique()]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Train shape: {df_full_train.shape}")
print(f"Val shape:   {df_full_val.shape}")
print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Train shape: (5952, 323)
Val shape:   (5964, 323)
Dummy subcat names: ['Flats']
All time steps: ['2025-04-28', '2025-05-26', '2025-06-23', '2025-07-21', '2025-08-18', '2025-09-15', '2025-10-13', '2025-11-10', '2025-12-08', '2026-01-05', '2026-02-02', '2026-03-02']


Index(['ASIN', 'date', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=323)

## ⑤ Sanity Check — Row Counts

In [5]:
print(f"Val rows:   {len(df_full_val)}")
print(f"Train rows: {len(df_full_train)}")

Val rows:   5964
Train rows: 5952


## ⑥ Define Controls and Feature Specifications

Same controls as 03_2 plus `controls_emb` — all 256 multimodal embedding dimensions.

In [6]:
n_lags = 1

outcome   = "Q_t"
treatment = "P_bb_t"

outcome_diff   = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls
additional_controls_deep = [var for var in additional_controls if var not in dummy_subcat_names]

# Key difference from 03_2 — multimodal emb columns
controls_emb = [var for var in df_full_train.columns if "emb" in var]

print(f"Continuous controls:    {len(cont_controls)}")
print(f"Dummy controls:         {len(dummy_controls)}")
print(f"Total controls:         {len(additional_controls)}")
print(f"Embedding columns:      {len(controls_emb)}")

Continuous controls:    5
Dummy controls:         13
Total controls:         18
Embedding columns:      256


## ⑦ Initialize Results DataFrames

In [7]:
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df      = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)
print('✅ Results DataFrames initialized')

✅ Results DataFrames initialized


## ⑧ Deep Model R² — Level

In [8]:
df_dict = {"Train": df_full_train, "Test": df_full_val}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome (time independent):   {r2_ml_l}")
    print(f"  R2 Treatment (time independent):  {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"  R2 Outcome (lag1):                {r2_ml_l_lag1}")
    print(f"  R2 Treatment (lag1):              {r2_ml_m_lag1}")
    print()

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
  R2 Outcome (time independent):   0.8598
  R2 Treatment (time independent):  0.769
  R2 Outcome (lag1):                0.9278
  R2 Treatment (lag1):              0.7622

Evaluation for Test set
  R2 Outcome (time independent):   0.7948
  R2 Treatment (time independent):  0.6661
  R2 Outcome (lag1):                0.9178
  R2 Treatment (lag1):              0.6588



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.8598,0.7948,0.7690,0.6661
Deep Time Dependent,0.9278,0.9178,0.7622,0.6588


## ⑨ Deep Model R² — Diff

In [9]:
results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome diff (time independent):   {r2_ml_l_diff}")
    print(f"  R2 Treatment diff (time independent):  {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"  R2 Outcome diff (lag1):                {r2_ml_l_diff_lag1}")
    print(f"  R2 Treatment diff (lag1):              {r2_ml_m_diff_lag1}")
    print()

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
  R2 Outcome diff (time independent):   0.11039544
  R2 Treatment diff (time independent):  -0.03369158
  R2 Outcome diff (lag1):                0.08684958
  R2 Treatment diff (lag1):              -0.00663046

Evaluation for Test set
  R2 Outcome diff (time independent):   0.09565223
  R2 Treatment diff (time independent):  -0.03813664
  R2 Outcome diff (lag1):                0.07199074
  R2 Treatment diff (lag1):              -0.00849455



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.110395,0.095652,-0.033692,-0.038137
Deep Time Dependent,0.086850,0.071991,-0.006630,-0.008495


## ⑩ Build Feature Matrices

Key difference from 03_2: adds `x_train_emb` and `x_test_emb`
using all 256 multimodal embedding dimensions.

In [10]:
# Train set
y_train      = df_full_train[outcome].squeeze()
d_train      = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train     = sm.add_constant(df_full_train[additional_controls])
x_train_pca = sm.add_constant(df_full_train[additional_controls + controls_pca])
x_train_sim = sm.add_constant(df_full_train[additional_controls + controls_similarities])
x_train_emb = sm.add_constant(df_full_train[additional_controls + controls_emb])

# Test set
y_test      = df_full_val[outcome].squeeze()
d_test      = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test     = sm.add_constant(df_full_val[additional_controls])
x_test_pca = sm.add_constant(df_full_val[additional_controls + controls_pca])
x_test_sim = sm.add_constant(df_full_val[additional_controls + controls_similarities])
x_test_emb = sm.add_constant(df_full_val[additional_controls + controls_emb])

# Rename columns for LightGBM
for df in [x_train, x_test, x_train_pca, x_test_pca,
           x_train_sim, x_test_sim, x_train_emb, x_test_emb]:
    df.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x), inplace=True)

print(f"x_train shape:     {x_train.shape}")
print(f"x_train_pca shape: {x_train_pca.shape}")
print(f"x_train_sim shape: {x_train_sim.shape}")
print(f"x_train_emb shape: {x_train_emb.shape}")

x_train shape:     (5952, 18)
x_train_pca shape: (5952, 23)
x_train_sim shape: (5952, 23)
x_train_emb shape: (5952, 274)


## ⑪ Build Dict Structures

In [11]:
train_dict = {
    "y": y_train, "y_diff": y_train_diff,
    "d": d_train, "d_diff": d_train_diff,
    "x": x_train, "x_pca": x_train_pca,
    "x_sim": x_train_sim, "x_emb": x_train_emb,
}

test_dict = {
    "y": y_test, "y_diff": y_test_diff,
    "d": d_test, "d_diff": d_test_diff,
    "x": x_test, "x_pca": x_test_pca,
    "x_sim": x_test_sim, "x_emb": x_test_emb,
}
print('✅ Train and test dicts ready')

✅ Train and test dicts ready


## ⑫ Tabular Models — Level

Four feature specifications including the new x_emb (Tabular + full embeddings).

In [12]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y"])
    print(f"  R2 Outcome train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d"])
    print(f"  R2 Treatment train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_tab = pd.concat([results_df_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_tab.columns = column_names
results_df_tab.index   = row_names
results_df_tab


 Feature specification: x
  OLS


  R2 Outcome train/test: 0.4587 / 0.3561
  R2 Treatment train/test: 0.1048 / 0.0952
  Boosting


  R2 Outcome train/test: 0.8234 / 0.7087


  R2 Treatment train/test: 0.6365 / 0.4269

 Feature specification: x_pca
  OLS
  R2 Outcome train/test: 0.8116 / 0.6987
  R2 Treatment train/test: 0.6324 / 0.5366
  Boosting


  R2 Outcome train/test: 0.9714 / 0.8724


...

  R2 Outcome train/test: 0.9190 / 0.7831


  R2 Treatment train/test: 0.8325 / 0.5038
  Boosting


  R2 Outcome train/test: 0.9810 / 0.8783


  R2 Treatment train/test: 0.9406 / 0.6737


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.458693,0.356116,0.104762,0.095230
Boosting (Tabular),0.823358,0.708723,0.636489,0.426913
OLS (Tabular + PCA),0.811614,0.698735,0.632354,0.536577
Boosting (Tabular + PCA),0.971364,0.872404,0.905885,0.692772
OLS (Tabular + Similarities),0.827604,0.725236,0.640049,0.537419
Boosting (Tabular + Similarities),0.964602,0.871729,0.888830,0.630209
OLS (Tabular + Embeddings),0.918990,0.783094,0.832476,0.503766
Boosting (Tabular + Embeddings),0.981010,0.878320,0.940599,0.673749


## ⑬ Summary — Level Models

In [13]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.458693,0.356116,0.104762,0.095230
Boosting (Tabular),0.823358,0.708723,0.636489,0.426913
OLS (Tabular + PCA),0.811614,0.698735,0.632354,0.536577
Boosting (Tabular + PCA),0.971364,0.872404,0.905885,0.692772
OLS (Tabular + Similarities),0.827604,0.725236,0.640049,0.537419
Boosting (Tabular + Similarities),0.964602,0.871729,0.888830,0.630209
OLS (Tabular + Embeddings),0.918990,0.783094,0.832476,0.503766
Boosting (Tabular + Embeddings),0.981010,0.878320,0.940599,0.673749
Deep Time Independent,0.859800,0.794800,0.769000,0.666100
Deep Time Dependent,0.927800,0.917800,0.762200,0.658800


## ⑭ Tabular Models — Diff

In [14]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y_diff"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome diff train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d_diff"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d_diff"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment diff train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y_diff"])
    print(f"  R2 Outcome diff train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d_diff"])
    print(f"  R2 Treatment diff train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_diff_tab = pd.concat([results_df_diff_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_diff_tab.columns = column_names
results_df_diff_tab.index   = row_names
results_df_diff_tab


 Feature specification: x
  OLS
  R2 Outcome diff train/test: 0.3503 / 0.3528
  R2 Treatment diff train/test: 0.0208 / 0.0093
  Boosting


  R2 Outcome diff train/test: 0.6655 / 0.5730


  R2 Treatment diff train/test: 0.2269 / 0.0128

 Feature specification: x_pca
  OLS
  R2 Outcome diff train/test: 0.3516 / 0.3537
  R2 Treatment diff train/test: 0.0219 / 0.0102
  Boosting


  R2 Outcome diff train/test: 0.7514 / 0.6153


  R2 Treatment diff train/test: 0.3731 / 0.0123

 Feature specification: x_sim
  OLS
  R2 Outcome diff train/test: 0.3516 / 0.3541
  R2 Treatment diff train/test: 0.0222 / 0.0112
  Boosting


...

  R2 Outcome diff train/test: 0.4132 / 0.2551


  R2 Treatment diff train/test: 0.0636 / -0.1852
  Boosting


  R2 Outcome diff train/test: 0.8147 / 0.6158


  R2 Treatment diff train/test: 0.4543 / -0.0662


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.350289,0.352760,0.020826,0.009250
Boosting (Tabular),0.665490,0.572952,0.226882,0.012775
OLS (Tabular + PCA),0.351642,0.353675,0.021868,0.010237
Boosting (Tabular + PCA),0.751377,0.615280,0.373086,0.012307
OLS (Tabular + Similarities),0.351650,0.354091,0.022217,0.011182
Boosting (Tabular + Similarities),0.722938,0.584173,0.338068,0.022418
OLS (Tabular + Embeddings),0.413207,0.255142,0.063595,-0.185204
Boosting (Tabular + Embeddings),0.814686,0.615801,0.454332,-0.066166


## ⑮ Summary — Diff Models

In [15]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.350289,0.352760,0.020826,0.009250
Boosting (Tabular),0.665490,0.572952,0.226882,0.012775
OLS (Tabular + PCA),0.351642,0.353675,0.021868,0.010237
Boosting (Tabular + PCA),0.751377,0.615280,0.373086,0.012307
OLS (Tabular + Similarities),0.351650,0.354091,0.022217,0.011182
Boosting (Tabular + Similarities),0.722938,0.584173,0.338068,0.022418
OLS (Tabular + Embeddings),0.413207,0.255142,0.063595,-0.185204
Boosting (Tabular + Embeddings),0.814686,0.615801,0.454332,-0.066166
Deep Time Independent,0.110395,0.095652,-0.033692,-0.038137
Deep Time Dependent,0.086850,0.071991,-0.006630,-0.008495


## ⑯ Final Summary (% format)

R² multiplied by 100 — matches paper Table 2 format.
Compare with 03_2 (txt only) to see the gain from adding image embeddings.

In [16]:
print("=== Level Models — Test R² (%) ===")
print(results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100)
print()
print("=== Diff Models — Test R² (%) ===")
print(results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100)

=== Level Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          35.61       9.52
Boosting (Tabular)                     70.87      42.69
OLS (Tabular + PCA)                    69.87      53.66
Boosting (Tabular + PCA)               87.24      69.28
OLS (Tabular + Similarities)           72.52      53.74
Boosting (Tabular + Similarities)      87.17      63.02
OLS (Tabular + Embeddings)             78.31      50.38
Boosting (Tabular + Embeddings)        87.83      67.37
Deep Time Independent                  79.48      66.61
Deep Time Dependent                    91.78      65.88

=== Diff Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          35.28       0.93
Boosting (Tabular)                     57.30       1.28
OLS (Tabular + PCA)                    35.37       1.02
Boosting (Tabular + PCA)               61.53       1.23
OLS (Tabular + Similarities)      